# 04 · Optimización del patrimonio con CVaR (Caso C)

Muestra la capa (iv) (`src/optimum/optimizer.py`). El problema es
$$\max_{x,y}\; E[\Delta PN] - \lambda\,\mathrm{CVaR}_\alpha(-\Delta PN) - TC$$
sujeto a los presupuestos, los límites del Cuadro 4 y las restricciones comunes (formulación §4-§6). Se resuelve como LP de Rockafellar–Uryasev con los escenarios del notebook 03.

Todas las cifras salen de `data/json/output.json` (`make run`, D-19). El notebook solo lee, muestra y exporta figuras y tablas; al final recalcula los escenarios para verificar que el output es reproducible. Decisiones: D-08 (costos), D-18 (caja al horizonte), D-19 (bloque `analysis`), D-20 (límites al horizonte).

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import numpy as np
import pandas as pd
from matplotlib import colormaps

from optimum import optimizer as opt
from optimum import paths
from optimum.io.json_contract import load_input, validate_output
from optimum.reporting.figures import COLORS, new_figure, save_fig
from optimum.reporting.labels import constraint_label, horizon_mode
from optimum.reporting.tables import save_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

doc = load_input()
out = json.loads(paths.OUTPUT_JSON.read_text(encoding="utf-8"))
assert validate_output(out) == [] and out["status"] == "OPTIMAL", out["status"]
assert out["caseId"] == doc["caseId"]
assert out["riskAversionLambda"] == doc["caseParameters"]["riskAversionLambda"], "output.json desactualizado: make run"

an = out["analysis"]
lam = out["riskAversionLambda"]
alpha = out["metrics"]["cvarAlpha"]
tag = f"{alpha * 100:g}"
inst = pd.DataFrame(doc["instruments"]).set_index("id")
print(f"Caso {out['caseId']} · status {out['status']} · λ = {lam} · α = {alpha} · solver {out['solver']['name']} "
      f"({out['solver']['iterations']} iteraciones, {out['solver']['time']:.3f} s)")
print(f"Límites de vencimiento al horizonte: {out['horizonMaturityLimits']} (variante: {an['horizonMaturityVariant']['mode']})")

## Posiciones: inicial, óptima y alternativas

- **Óptimo:** λ = `riskAversionLambda`, límites del Cuadro 4 en $t_0$ (lectura literal del enunciado).
- **Media-varianza:** mínima varianza con el mismo $E[\Delta PN] - TC$ que el óptimo CVaR (§9.3).
- **Variante de vencimientos:** además impone LE_12M y GT_36M medidos desde $t_H$ (D-20).

In [ ]:
mv = an["meanVarianceBenchmark"]
hv = an["horizonMaturityVariant"]
label_opt, label_mv, label_hv = f"Óptimo (λ = {lam:g})", "Media-varianza", "Venc. exigidos al horizonte"

rows = out["positions"]["assets"] + out["positions"]["liabilities"]
pos = pd.DataFrame(rows).set_index("id")
pos_tab = pd.DataFrame(
    {
        "Lado": np.where(pos.index.isin([r["id"] for r in out["positions"]["assets"]]), "Activo", "Pasivo"),
        "Instrumento": pos["name"],
        "Moneda": pos["currency"],
        "Inicial": pos["initial"],
        label_opt: pos["optimal"],
        "Cambio": pos["change"],
        label_mv: pd.Series(mv["positions"]),
        label_hv: pd.Series(hv["positions"]),
    }
)
save_table(
    pos_tab,
    "posiciones_optimas",
    caption=f"Posiciones inicial, óptima (λ = {lam:g}) y alternativas (S/ mm).",
    label="tab:posiciones-optimas",
    fmt="{:,.1f}",
)
pos_tab.round(1)

In [ ]:
fig, axes = new_figure(1, 2, aspect=0.42)
for ax, side, title in zip(axes, ["Activo", "Pasivo"], ["Activos", "Pasivos"], strict=True):
    t = pos_tab[pos_tab["Lado"] == side]
    k = np.arange(len(t))
    w = 0.38
    ax.bar(k - w / 2, t["Inicial"], w, color=COLORS["neutral"], alpha=0.45, label="Inicial")
    ax.bar(k + w / 2, t[label_opt], w, color=COLORS["ASSET" if side == "Activo" else "LIABILITY"], label=label_opt)
    ax.scatter(k + w / 2, t[label_hv], marker="_", s=180, color="black", zorder=3, label=label_hv)
    ax.set_xticks(k, [f"{i}\n{c}" for i, c in zip(t.index, t["Moneda"], strict=True)], fontsize=7)
    ax.set_title(title)
axes[0].set_ylabel("S/ mm")
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.16))
_ = save_fig(fig, "posiciones_inicial_optima")

## Métricas

$E[PN_T] = PN_0 - TC + E[\Delta PN]$. VaR y CVaR de la pérdida $-\Delta PN$, recalculados ex post con átomo fraccional (§5). El objetivo de cada columna se evalúa con la λ de referencia. Pérdida en estrés > 0 significa que el patrimonio cae.

In [ ]:
def metric_rows(m: dict, objective: float) -> dict:
    return {
        "Objetivo (E − λ·CVaR − TC)": objective,
        "E[ΔPN]": m["expectedNetWorthChange"],
        "Costo de transacción": m["transactionCosts"],
        "E[ΔPN] − TC": m["expectedNetWorthChange"] - m["transactionCosts"],
        "E[PN] al horizonte": m["expectedTerminalNetWorth"],
        "σ(ΔPN)": m["netWorthVolatility"],
        f"VaR {tag} %": m[f"var{tag}Loss"],
        f"CVaR {tag} %": m[f"cvar{tag}Loss"],
        "Rentabilidad esperada activos (%)": 100 * m["expectedAssetReturn"],
        "Costo esperado pasivos (%)": 100 * m["expectedLiabilityEconomicCost"],
        "Estrés en la cola (%)": 100 * m["tailStressShare"],
        **{f"Pérdida {k}": v for k, v in m["stressLosses"].items()},
    }


ini = an["initialPosition"]
met = pd.DataFrame(
    {
        "Inicial": metric_rows(ini["metrics"], ini["objective"]),
        label_opt: metric_rows(out["metrics"], out["objectiveValue"]),
        label_mv: metric_rows(mv["metrics"], mv["objective"]),
        label_hv: metric_rows(hv["metrics"], hv["objective"]),
    }
)
save_table(
    met,
    "metricas_optimizacion",
    caption=f"Métricas a un año (S/ mm salvo indicación). α = {alpha:g}; objetivo evaluado con λ = {lam:g}.",
    label="tab:metricas-optimizacion",
    fmt="{:,.2f}",
)
print("Límites que incumple la posición inicial:", ini["violatedConstraints"])
print(f"Costo de imponer los vencimientos en t_H: {hv['objectiveCost']:.3f} S/ mm de objetivo")
met.round(2)

## Restricciones en el óptimo

Pesos como fracción del total del lado post-decisión (presupuestos fijos). *Activa* = en su cota (tolerancia relativa 1e-7). Los grupos vacíos no son vinculantes.

In [ ]:
pretty = constraint_label


chk = pd.DataFrame(out["constraintChecks"])
chk_tab = pd.DataFrame(
    {
        "Valor": chk["value"],
        "Mín.": chk["min"],
        "Máx.": chk["max"],
        "Activa": np.where(chk["active"], "sí", ""),
        "Nota": chk["note"],
    }
)
chk_tab.index = chk["name"].map(pretty)
assert chk["ok"].all()
save_table(
    chk_tab,
    "restricciones_optimo",
    caption="Restricciones en el óptimo: pesos (fracción del lado), montos (S/ mm) y límites activos.",
    label="tab:restricciones-optimo",
    fmt="{:,.3f}",
)
print(f"{chk['active'].sum()} restricciones activas de {len(chk)}")

# Pesos por moneda, tipo, tasa y vencimiento (estadística mínima del enunciado) + concentraciones activas
dims = chk["name"].str.split(":")
is_group = dims.str.len().ge(3) & dims.str[1].isin(["CURRENCY", "TYPE", "RATE_TYPE", "MATURITY"])
# Concentraciones solo si están en su tope: una posición en 0 toca el mínimo (0 %) y ya figura como no negatividad
is_conc = dims.str[1].eq("CONCENTRATION") & chk["active"] & (chk["value"] >= chk["max"].astype(float) - 1e-6)
wt = chk[is_group | is_conc]
zero = [q["id"] for q in out["positions"]["assets"] + out["positions"]["liabilities"] if abs(q["optimal"]) < 1e-6]
nonneg = f"la no negatividad ({', '.join(zero)} = 0) y " if zero else ""
wt_tab = pd.DataFrame(
    {
        "Peso (%)": 100 * wt["value"].to_numpy(),
        "Mín. (%)": 100 * wt["min"].astype(float).to_numpy(),
        "Máx. (%)": 100 * wt["max"].astype(float).to_numpy(),
        "Activa": np.where(wt["active"], "sí", ""),
    },
    index=wt["name"].map(pretty),
)
save_table(
    wt_tab,
    "pesos_optimo",
    caption=f"Pesos de la cartera óptima (λ = {lam:g}) por moneda, tipo, tasa y vencimiento (\\% del lado) y "
    f"concentraciones en su tope. Además están activas {nonneg}los presupuestos.",
    label="tab:pesos-optimo",
    fmt="{:,.1f}",
)
chk_tab

## Barrido de aversión al riesgo (`lambdaGrid`)

Por preferencia revelada, al subir λ no crecen ni el CVaR ni $E[\Delta PN] - TC$; $E[\Delta PN]$ sola puede subir si el optimizador reestructura más (y paga más TC).

In [ ]:
def w(s, dim, side, cat):
    return 100 * s["weights"][dim][side].get(cat, 0.0)


sweep = pd.DataFrame(
    [
        {
            "λ": s["lambda"],
            "E[ΔPN]": s["metrics"]["expectedNetWorthChange"],
            "TC": s["metrics"]["transactionCosts"],
            "E[ΔPN] − TC": s["metrics"]["expectedNetWorthChange"] - s["metrics"]["transactionCosts"],
            "σ(ΔPN)": s["metrics"]["netWorthVolatility"],
            f"CVaR {tag} %": s["metrics"][f"cvar{tag}Loss"],
            "Activos USD (%)": w(s, "byCurrency", "ASSET", "USD"),
            "Equity (%)": w(s, "byType", "ASSET", "EQUITY"),
            "Pasivos USD (%)": w(s, "byCurrency", "LIABILITY", "USD"),
            "Pasivos flotantes (%)": w(s, "byRateType", "LIABILITY", "FLOAT"),
        }
        for s in an["lambdaSweep"]
    ]
)
assert (sweep[f"CVaR {tag} %"].diff().dropna() <= 1e-5).all()
assert (sweep["E[ΔPN] − TC"].diff().dropna() <= 1e-5).all()
sweep_tab = sweep.assign(λ=sweep["λ"].map(lambda v: f"{v:g}"))
save_table(
    sweep_tab,
    "barrido_lambda",
    caption="Barrido de λ: resultado esperado, costo, riesgo y composición (S/ mm y \\% del lado).",
    label="tab:barrido-lambda",
    fmt="{:,.2f}",
    index=False,
)
sweep = sweep.set_index("λ")
sweep.round(2)

In [ ]:
fig, ax = new_figure(aspect=0.5)
ax.plot(sweep[f"CVaR {tag} %"], sweep["E[ΔPN] − TC"], "-o", color=COLORS["ASSET"], ms=4, label="Frontera CVaR (λ)")
for lam_k, r in sweep.iterrows():
    ax.annotate(f"λ={lam_k:g}", (r[f"CVaR {tag} %"], r["E[ΔPN] − TC"]), textcoords="offset points", xytext=(4, 4), fontsize=7)
points = [
    ("Inicial", ini["metrics"], COLORS["neutral"], "s"),
    (label_mv, mv["metrics"], COLORS["USD"], "D"),
    (label_hv, hv["metrics"], COLORS["LIABILITY"], "^"),
]
for name, m, color, marker in points:
    ax.scatter(m[f"cvar{tag}Loss"], m["expectedNetWorthChange"] - m["transactionCosts"], color=color, marker=marker, s=40, zorder=3, label=name)
ax.axhline(0, color="black", lw=0.6)
ax.set(xlabel=f"CVaR {tag} % de la pérdida (S/ mm)", ylabel="E[ΔPN] − TC (S/ mm)", title="Frontera resultado esperado neto vs. CVaR")
ax.legend(fontsize=7, loc="lower right")
_ = save_fig(fig, "frontera_lambda")

In [ ]:
fig, axes = new_figure(1, 2, aspect=0.45)
lams = [s["lambda"] for s in an["lambdaSweep"]]
comp = pd.DataFrame([s["positions"] for s in an["lambdaSweep"]], index=[f"{v:g}" for v in lams])
cmaps = {"PEN": "Blues", "USD": "Oranges"}
for ax, side, title in zip(axes, ["ASSET", "LIABILITY"], ["Activos", "Pasivos"], strict=True):
    ids = inst.index[inst["side"] == side]
    bottom = np.zeros(len(comp))
    for ccy in ["PEN", "USD"]:
        sub = [i for i in ids if inst.loc[i, "currency"] == ccy]
        shades = colormaps[cmaps[ccy]](np.linspace(0.45, 0.9, len(sub)))
        for i, c in zip(sub, shades, strict=True):
            ax.bar(comp.index, comp[i], bottom=bottom, color=c, label=f"{i} ({ccy})", width=0.7)
            bottom += comp[i].to_numpy()
    ax.set(title=title, xlabel="λ")
    ax.legend(fontsize=6, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
axes[0].set_ylabel("S/ mm")
_ = save_fig(fig, "composicion_lambda")

## Incumplimientos al horizonte (§7.5, D-18, D-20)

Se reportan, no se imponen. Balance en $t_H$: tenencias revalorizadas; los flujos netos del año y el TC pagado en $t_0$ quedan en caja. *Estructural*: el grupo se incumple aun con los montos de $t_0$ (lo provoca el paso del tiempo sobre la decisión); *mercado*: solo por la deriva de los valores.

In [ ]:
def breaches_table(m: dict) -> pd.DataFrame:
    com = doc["constraints"]["common"]
    limits = {
        "minNetWorth": ("PN al horizonte (S/ mm)", com["minNetWorth"], None),
        "maxLiabilitiesToAssets": ("P/A al horizonte", None, com["maxLiabilitiesToAssets"]),
        "minCash": ("Caja al horizonte (S/ mm)", com["minCash"], None),
    }
    rows = {}
    for k, v in m["horizonBreaches"].items():
        name, lo, hi = limits.get(k, (pretty(k), v.get("min"), v.get("max")))
        if v["count"] > 0:  # los límites sin incumplimientos se mencionan en el texto
            rows[name] = {"Prob. (%)": 100 * v["probability"], "Peor valor": v["worst"], "Mín.": lo, "Máx.": hi, "Causa": v.get("cause") or ""}
    t = pd.DataFrame(rows).T
    num = ["Prob. (%)", "Peor valor", "Mín.", "Máx."]
    t[num] = t[num].astype(float)
    return t


hz = breaches_table(out["metrics"])
hz_var = breaches_table(hv["metrics"])
hz["Prob. venc. exigidos (%)"] = hz_var["Prob. (%)"].reindex(hz.index).fillna(0.0)
for name in hz_var.index.difference(hz.index):  # incumplimientos que solo aparecen en la variante
    hz.loc[name] = {**hz_var.loc[name], "Prob. (%)": 0.0, "Prob. venc. exigidos (%)": hz_var.loc[name, "Prob. (%)"]}
save_table(
    hz,
    "incumplimientos_horizonte",
    caption=f"Incumplimientos al horizonte de la solución óptima (λ = {lam:g}) y de la variante con "
    f"{horizon_mode(hv['mode'])}. PN, P/A y caja no se incumplen en ningún escenario.",
    label="tab:incumplimientos-horizonte",
    fmt="{:,.3f}",
)
hz

## Distribución de ΔPN y control de reproducibilidad

Se regeneran los escenarios y $r_{k,s}$ desde `input.json` (≈ 50 s) y se reevalúan las posiciones de `output.json`: las métricas deben coincidir.

In [ ]:
data = opt.prepare(doc)  # ≈ 50 s
positions = {
    "Inicial": data.x0,
    label_opt: pos["optimal"],
    label_mv: pd.Series(mv["positions"]),
}
ev = {k: opt.evaluate(data, v) for k, v in positions.items()}
assert abs(ev[label_opt]["cvar"] - out["metrics"][f"cvar{tag}Loss"]) < 1e-6
assert abs(ev[label_opt]["expected_pnl"] - out["metrics"]["expectedNetWorthChange"]) < 1e-6
assert abs(ev["Inicial"]["cvar"] - ini["metrics"][f"cvar{tag}Loss"]) < 1e-6
print("output.json reproducido: E[ΔPN] y CVaR coinciden con tolerancia 1e-6")

r = data.returns.to_numpy()
kind = np.asarray(data.scenario_kind)
fig, ax = new_figure(aspect=0.45)
styles = {"Inicial": COLORS["neutral"], label_opt: COLORS["ASSET"], label_mv: COLORS["USD"]}
dpns = {name: (data.sign * p.reindex(data.ids).to_numpy()) @ r for name, p in positions.items()}
lo, hi = min(d.min() for d in dpns.values()), max(d.max() for d in dpns.values())
bins = np.linspace(np.floor(lo), np.ceil(hi), 61)
print(f"ΔPN en [{lo:.1f}, {hi:.1f}] S/ mm")
for name, dpn in dpns.items():
    ax.hist(dpn[kind == "bootstrap"], bins=bins, histtype="step", lw=1.3, color=styles[name], label=name)
    ax.scatter(dpn[kind == "stress"], np.full((kind == "stress").sum(), -2.0 if name == "Inicial" else -5.0 if name == label_opt else -8.0),
               marker="v", s=28, color=styles[name], zorder=3)
    ax.axvline(-ev[name]["cvar"], color=styles[name], ls=":", lw=1)
ax.set(xlabel="ΔPN a un año (S/ mm)", ylabel="Escenarios bootstrap", title="Distribución de ΔPN (▼ estrés; punteada: −CVaR)")
ax.legend(fontsize=7)
_ = save_fig(fig, "distribucion_dpn_optima")

## Lectura

> Borrador redactado por la IA a partir de las salidas de este notebook y de la auditoría del optimizador (D-20). Lo revisa y valida el alumno.

**1. Qué hace el óptimo.** Con λ = 0.25 se reestructuran ambos lados y 10 de las 29 restricciones quedan activas, algo esperable en un LP.
- **Activos:** el equity A06 baja de 220 a 50, su piso implícito: con la caja ≤ 20 % y la renta fija ≤ 75 %, el equity no puede bajar de 5 %. La caja sube al tope de 200 y A02 al tope de concentración (350). A04 sale por completo porque A05 lo domina: tiene más media (6.32 % frente a 6.23 %) y menos σ (5.61 % frente a 6.74 %), según la tabla `rendimientos_escenarios` del notebook 03.
- **Pasivos:** L01 sube al tope de 280 porque es el pasivo más barato (5.73 %) y el menos volátil. L02 baja de 150 a 40, porque ahorra 1.74 %/año (7.47 % − 5.73 %) frente a un costo de 1.25 % por salir de L02 y entrar en L01. L06 baja a 50.
- **Límites:** la posición inicial solo incumplía el de equity (22 % > 20 %).

**2. Riesgo contra valor esperado.**
- **Riesgo:** el CVaR 95 % cae de 50.1 a 7.8 S/ mm y la σ(ΔPN) de 25.4 a 7.0.
- **Valor esperado:** E[ΔPN] sube de 3.96 a 5.83, pero el costo de transacción es 2.72. Por eso E[ΔPN] − TC = 3.11 queda por debajo del 3.96 de la posición inicial. El óptimo **cede 0.85 S/ mm de valor esperado a cambio de bajar el CVaR en 42 S/ mm**. Así hay que presentarlo, y no como una mejora de rentabilidad.

**3. Estrés.** El óptimo pierde 10.4 S/ mm en C_STRESS_2, que es el único estrés que entra en la cola: explica el 25 % de su masa (0.0125 / 0.05). En los otros tres estrés gana entre 17.7 y 24.2.
- **Cautela (auditoría I-1, precisada en la auditoría de sensibilidades, D-23):** esas ganancias dependen de la regla de cupones flotantes (D-13). Con la regla de forward por período (S9), **esta misma cartera** gana 5.8 en C_STRESS_3 (no 18.8) y 23.2 en C_STRESS_4 (no 24.2); la cartera **reoptimizada** con esa regla (L01 230, L06 100) gana 0.9 y 15.4. Las posiciones de activos no cambian al reoptimizar, pero su resultado en estrés sí: el mayor cambio en C_STRESS_3 viene de A05 (−25.8). La ganancia en C_STRESS_4 es robusta; la de C_STRESS_3, no. Detalle en el notebook 05.

**4. Frontera λ.** La curva muestra rendimientos decrecientes al reducir riesgo.
- **Entre λ = 0 y λ = 0.1:** el CVaR baja de 61 a 32 S/ mm a cambio de 0.75 de E − TC.
- **Entre λ = 0.1 y λ = 0.25:** baja a 7.8 a cambio de 3.0 más.
- **Desde λ = 0.5:** reducir el CVaR de 2.5 a 1.35 cuesta 1.8 más, y con λ ≥ 2 el resultado neto esperado es ≤ 0, peor que no reestructurar.
- **Por qué E[ΔPN] sube con λ ≥ 0.5:** el optimizador recompra L03 (el pasivo más caro, 8.36 %) y aumenta L05 para calzar monedas. Paga entre 6 y 9 S/ mm de costo de transacción, así que E[ΔPN] sola no es monótona pero E − TC sí.
- **Conclusión:** λ = 0.25 queda en el codo de la curva.

**5. Media-varianza.** Con el mismo E − TC (3.11), el portafolio de mínima varianza tiene menos σ (6.40 frente a 7.03) y menos VaR (4.23 frente a 5.26). En cambio tiene **más CVaR** (9.23 frente a 7.82) y pierde 17.7 S/ mm en C_STRESS_2, frente a 10.4. La varianza penaliza por igual las ganancias y las pérdidas y diluye el estrés, que pesa poco en el segundo momento. El CVaR apunta justamente a la cola que pide el Caso C.

**6. Horizonte.**
- **Restricciones comunes:** PN, P/A y caja nunca se incumplen al horizonte (el peor PN_T es 185 frente al mínimo de 120).
- **LE_12M:** se incumple con probabilidad 1 y es **estructural**. L01 vence el 2027-12-31, así que al horizonte le quedan 12 meses. La variante que impone los vencimientos medidos desde $t_H$ cuesta 0.287 S/ mm de objetivo: deja L01 en 200 y reparte la diferencia entre L02, L03 y L06. Además baja el CVaR a 7.33 y la σ a 5.51, con E − TC = 2.70.
- **Límites que se deslizan por el mercado:** renta fija 68 %, A02 53 %, L01 36 %, fija/flotante 34 % y caja 18 % de probabilidad. Una solución que está en el borde de sus límites los cruza con cualquier revalorización favorable. No lo causa la decisión, pero la gestión tendría que rebalancear durante el año. Hasta la variante sigue incumpliendo LE_12M con 36 % de probabilidad por deriva de mercado, porque queda justo en 25 %.

**7. Reproducibilidad.** Se regeneraron los escenarios desde `input.json` y E[ΔPN] y el CVaR coinciden con `output.json` con tolerancia 1e-6.